# Harvest Slot DL 학습 파이프라인

이 노트북은 **노트북 파일 1개 + Kaggle에 연결한 데이터셋**만으로 실행되도록 구성한 self-contained 학습 문서입니다.

외부 `ai/dl/freshness/*.py` 파일을 import하지 않습니다. 데이터 처리, Dataset/DataLoader, ResNet18 모델, 학습 루프, checkpoint 선정, 추론, FastAPI 응답 변환에 필요한 코드를 모두 노트북 안에 포함합니다.


## 전체 흐름

1. Kaggle/로컬 실행 환경과 GPU 확인
2. 데이터 위치 자동 탐색
3. QC 원천 데이터 또는 준비된 ImageFolder 데이터 처리
4. 데이터 기준과 라벨 매핑 설명
5. 보조 이미지 특징과 신선도 점수 rule 정의
6. Dataset/DataLoader 구성
7. ResNet18 학습 및 best checkpoint 저장
8. 최종 모델 선정 기준 확인
9. 단일 이미지 추론
10. FastAPI 연동용 응답 변환 준비


In [ ]:
# 1단계: 실행 환경, 공통 라이브러리, GPU 상태를 확인합니다.
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import copy
import json
import random
import re
import zipfile
from typing import Any

import numpy as np
from PIL import Image

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, f1_score

IS_KAGGLE = Path("/kaggle").exists()
NOTE_DIR = Path.cwd()
PROJECT_ROOT = NOTE_DIR.parent if NOTE_DIR.name == "Note" else NOTE_DIR
WORKING_ROOT = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "ai" / "dl" / "freshness"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

print("IS_KAGGLE:", IS_KAGGLE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORKING_ROOT:", WORKING_ROOT)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 1. 기본 설정

이 노트북은 **과일 1종을 학습할 때마다 상단 설정만 바꿔 재사용**하는 구조입니다.

현재 원천 데이터셋은 보통 아래처럼 구성됩니다.

```text
Training/
  라벨링데이터/ 또는 원천데이터/
    품종별 zip 파일들
Validation/
  라벨링데이터/ 또는 원천데이터/
    품종별 zip 파일들
```

zip을 풀면 과일 안에 여러 품종이 있고, 각 품종 아래가 `L/M/S` 등급 폴더로 나뉘는 구조를 가정합니다.

참고1 PDF 기준으로 이 프로젝트에서 사용할 품목과 품종은 아래 4개 과일로 제한합니다.

| 과일 코드 | 품목 | 사용할 품종 |
|---|---|---|
| `apple` | 사과 | 부사, 양광 |
| `pear` | 배 | 신고, 추황 |
| `mandarine` | 감귤 | 한라봉, 온주밀감 |
| `persimmon` | 감 | 약시(반시), 대봉, 부유 |

이 노트북은 이미지 학습이 목적이므로 기본적으로 **원천데이터 이미지**를 사용합니다. JSON 라벨링 파일은 현재 A/B/C 폴더 라벨 학습에는 필수로 쓰지 않습니다.

최종적으로는 과일별 checkpoint가 1개씩 만들어지는 것을 목표로 합니다.

```text
apple_resnet18_best.pt
pear_resnet18_best.pt
mandarine_resnet18_best.pt
persimmon_resnet18_best.pt
```

On Kaggle, original images are read directly from /kaggle/input; only model checkpoints are saved under /kaggle/working.


In [ ]:
# 2단계: 과일, 데이터 탐색 규칙, 라벨, 학습 경로, 모델 저장 경로를 정의합니다.
# 과일별 학습 시 보통 TARGET_FRUIT만 바꾸면 됩니다.

# 서비스/모델에서 사용할 과일 코드입니다.
# 참고1 PDF 기준 사용 대상: "apple", "pear", "mandarine", "persimmon"
TARGET_FRUIT = "apple"

# 참고1 PDF 기준 품목/품종 설정입니다.
# 배는 배추/양배추와, 감은 감귤/감자와 이름이 겹치므로 exclude_keywords로 오탐을 줄입니다.
FRUIT_CONFIGS = {
    "apple": {
        "display_name": "사과",
        "varieties": ["부사", "양광", "fuji", "yanggwang"],
        "keywords": ["apple", "사과", "부사", "양광"],
        "exclude_keywords": [],
        "default_variety": "fuji",
    },
    "pear": {
        "display_name": "배",
        "varieties": ["신고", "추황", "singo", "chuhwang"],
        "keywords": ["pear", "배", "신고", "추황"],
        "exclude_keywords": ["배추", "양배추", "cabbage"],
        "default_variety": "singo",
    },
    "mandarine": {
        "display_name": "감귤",
        "varieties": ["한라봉", "온주밀감", "hallabong", "onjumilgam"],
        "keywords": ["mandarine", "감귤", "한라봉", "온주밀감"],
        "exclude_keywords": [],
        "default_variety": "hallabong",
    },
    "persimmon": {
        "display_name": "감",
        "varieties": ["반시", "대봉", "부유", "bansi", "daebong", "booyu"],
        "keywords": ["persimmon", "반시", "대봉", "부유"],
        "exclude_keywords": ["감귤", "감자", "mandarine", "potato"],
        "default_variety": "daebong",
    },
}

FRUIT_CONFIG = FRUIT_CONFIGS[TARGET_FRUIT]

# 품종 또는 데이터셋 세부 이름입니다.
# 특정 품종 하나만 학습하려면 SELECTED_VARIETIES에 품종명을 넣습니다.
TARGET_VARIETY = FRUIT_CONFIG["default_variety"]

# 비워두면 해당 과일의 참고1 PDF 대상 품종 전체를 합쳐 하나의 과일 모델을 학습합니다.
# 예: ["부사"] 또는 ["apple_fuji"]처럼 넣으면 특정 품종만 사용합니다.
SELECTED_VARIETIES: list[str] = []

# 기존 apple_fuji 같은 원천 폴더 구조도 지원하기 위한 설정입니다.
SOURCE_FOLDER_NAME = f"{TARGET_FRUIT}_{TARGET_VARIETY}"
SOURCE_LABEL_FOLDER_TEMPLATE = "{source_folder}_{source_label}"

# 파일명에서 같은 객체 ID를 추출하는 정규식입니다.
# 기본값은 apple_fuji_L_2-10_5DI90.png 같은 파일명에서 객체 ID 2를 추출합니다.
# 정규식이 맞지 않는 파일은 stem 전체를 객체 ID로 사용합니다.
OBJECT_ID_REGEX = r".*_[LMS]_(\d+)-\d+_"

# 공식 Training/Validation split이 있는 경우:
# - Training 데이터에서 train/valid를 다시 나눠 early stopping에 사용
# - Validation 데이터는 최종 test로 사용
USE_OFFICIAL_VALIDATION_AS_TEST = True
TRAIN_TO_VALID_RATIO = 0.1

LABELS = ["A", "B", "C"]
QC_LABEL_MAP = {"L": "A", "M": "B", "S": "C"}
QC_GRADE_MAP = {"A": "특", "B": "상", "C": "보통"}
DECISIONS = ["PASS", "REVIEW", "HOLD"]

DEFAULT_IMAGE_SIZE = 224
DEFAULT_MODEL_NAME = "resnet18"
DEFAULT_MODEL_VERSION = f"freshness-{TARGET_FRUIT}-resnet18-v0"

TRAIN_RATIO = 0.7
VALID_RATIO = 0.1
TEST_RATIO = 0.2
RANDOM_SEED = 42

EXTRACT_ROOT = WORKING_ROOT / "extracted-data"
PREPARED_DATA_ROOT = WORKING_ROOT / f"{TARGET_FRUIT}-freshness-data"
MODEL_ROOT = WORKING_ROOT / "models"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
ZIP_EXTS = {".zip"}

# Kaggle Add data 후 실제 데이터가 위치하는 루트입니다.
# 이 경로 아래에 training/validation 또는 Training/Validation 폴더가 있어야 합니다.
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/hmw0320/data-fruits")

print("대상 과일:", TARGET_FRUIT, "/", FRUIT_CONFIG["display_name"])
print("참고1 PDF 대상 품종:", FRUIT_CONFIG["varieties"])
print("선택 품종:", SELECTED_VARIETIES if SELECTED_VARIETIES else "대상 품종 전체 사용")
print("탐색 키워드:", FRUIT_CONFIG["keywords"])
print("제외 키워드:", FRUIT_CONFIG["exclude_keywords"])
print("객체 ID 정규식:", OBJECT_ID_REGEX)
print("공식 Validation을 test로 사용:", USE_OFFICIAL_VALIDATION_AS_TEST)
print("준비 데이터 출력 경로:", PREPARED_DATA_ROOT)
print("압축 해제 경로:", EXTRACT_ROOT)
print("Kaggle 데이터 루트:", KAGGLE_DATA_ROOT)
print("모델 저장 경로:", MODEL_ROOT)


## 2. Data Layout And Automatic Handling

This notebook assumes the Kaggle data is attached in one of these shapes.

### Option A: Already prepared ImageFolder layout

```text
/kaggle/input/datasets/hmw0320/data-fruits/
  train/{TARGET_FRUIT}/A
  train/{TARGET_FRUIT}/B
  train/{TARGET_FRUIT}/C
  valid/{TARGET_FRUIT}/A
  valid/{TARGET_FRUIT}/B
  valid/{TARGET_FRUIT}/C
  test/{TARGET_FRUIT}/A
  test/{TARGET_FRUIT}/B
  test/{TARGET_FRUIT}/C
```

### Option B: Official Training/Validation folders with raw image grade folders

```text
/kaggle/input/datasets/hmw0320/data-fruits/
  Training/
    source/raw image folders ... /L, /M, /S
  Validation/
    source/raw image folders ... /L, /M, /S
```

For Option B, the notebook no longer copies images into `/kaggle/working`. It builds an in-memory manifest of original image paths under `/kaggle/input` and trains a custom `Dataset` directly from those files.

Default split policy:

```text
Training source images -> train/valid
Validation source images -> test
```

Official Validation is kept as the final holdout, and a portion of Training is used as validation for early stopping.

Zip extraction is disabled on Kaggle by default (`ALLOW_KAGGLE_ZIP_EXTRACT = False`) because extracting large zip files can also exceed `/kaggle/working` capacity. Attach an extracted image dataset when running on Kaggle.


In [ ]:
# Step 3: Find source data roots without copying images into /kaggle/working.
# In Kaggle, raw image paths under /kaggle/input are used directly through an in-memory manifest.
OBJECT_ID_PATTERN = re.compile(OBJECT_ID_REGEX)
ALLOW_KAGGLE_ZIP_EXTRACT = False


def normalized_text(value: str) -> str:
    return value.lower().replace(" ", "").replace("_", "").replace("-", "").replace(".", "")


def contains_any_keyword(path: Path, keywords: list[str]) -> bool:
    target = normalized_text(str(path))
    return any(normalized_text(keyword) in target for keyword in keywords)


def contains_excluded_keyword(path: Path) -> bool:
    target = normalized_text(str(path))
    return any(normalized_text(keyword) in target for keyword in FRUIT_CONFIG.get("exclude_keywords", []))


def matches_target_fruit_or_variety(path: Path) -> bool:
    if contains_excluded_keyword(path):
        return False
    if contains_any_keyword(path, FRUIT_CONFIG.get("varieties", [])):
        return True
    return contains_any_keyword(path, FRUIT_CONFIG.get("keywords", [TARGET_FRUIT]))


def is_source_data_path(path: Path) -> bool:
    text = str(path).lower()
    return ("원천" in text) or ("source" in text) or ("raw" in text)


def is_training_path(path: Path) -> bool:
    text = str(path).lower()
    return ("training" in text) or ("train" in text) or ("학습" in text)


def is_validation_path(path: Path) -> bool:
    text = str(path).lower()
    return ("validation" in text) or ("valid" in text) or ("검증" in text)


def count_images_under(folder: Path) -> int:
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def find_child_dir_case_insensitive(parent: Path, name: str) -> Path | None:
    if not parent.exists():
        return None
    exact = parent / name
    if exact.exists() and exact.is_dir():
        return exact
    name_lower = name.lower()
    for child in parent.iterdir():
        if child.is_dir() and child.name.lower() == name_lower:
            return child
    return None


def find_descendant_dir_case_insensitive(root: Path, name: str) -> list[Path]:
    if not root.exists():
        return []
    name_lower = name.lower()
    return [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == name_lower]


def is_prepared_imagefolder(root: Path, fruit_type: str = TARGET_FRUIT) -> bool:
    return all((root / split / fruit_type / label).exists() for split in ["train", "valid", "test"] for label in LABELS)


def find_prepared_data_root(search_roots: list[Path]) -> Path | None:
    for root in search_roots:
        if not root.exists():
            continue
        if is_prepared_imagefolder(root):
            return root
        for candidate in root.rglob("train"):
            parent = candidate.parent
            if is_prepared_imagefolder(parent):
                return parent
    return None


def official_split_from_path(path: Path) -> str:
    if is_training_path(path):
        return "training"
    if is_validation_path(path):
        return "validation"
    return "unknown"


def extract_relevant_zips(search_roots: list[Path], extract_root: Path) -> list[Path]:
    """Extract source-data zip files only when extraction is explicitly allowed."""
    if IS_KAGGLE and not ALLOW_KAGGLE_ZIP_EXTRACT:
        print("Skipping zip extraction on Kaggle to protect /kaggle/working capacity.")
        return []

    zip_files: list[Path] = []
    for root in search_roots:
        if root.exists():
            zip_files.extend(p for p in root.rglob("*.zip") if p.is_file())

    relevant_zips = [
        p for p in zip_files
        if matches_target_fruit_or_variety(p)
        and (is_source_data_path(p) or not any("라벨" in part or "label" in part.lower() for part in p.parts))
    ]

    if not relevant_zips:
        print("No relevant source-data zip files were found for the selected fruit.")
        return []

    extracted_roots = []
    extract_root.mkdir(parents=True, exist_ok=True)
    for zip_path in sorted(relevant_zips):
        split_name = official_split_from_path(zip_path)
        target = extract_root / split_name / zip_path.stem
        marker = target / ".extract_done"
        if not marker.exists():
            target.mkdir(parents=True, exist_ok=True)
            print("extract:", zip_path, "->", target)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)
            marker.write_text("done", encoding="utf-8")
        extracted_roots.append(target)

    return extracted_roots


def find_grade_dirs(search_roots: list[Path]) -> list[tuple[Path, str, str]]:
    """Return tuples of (grade_dir, source_label, official_split)."""
    selected = [normalized_text(v) for v in SELECTED_VARIETIES]
    allowed_varieties = [normalized_text(v) for v in FRUIT_CONFIG.get("varieties", [])]
    result: list[tuple[Path, str, str]] = []

    for root in search_roots:
        if not root.exists():
            continue
        for folder in root.rglob("*"):
            if not folder.is_dir():
                continue
            source_label = folder.name[-1:].upper()
            if source_label not in QC_LABEL_MAP:
                continue
            if count_images(folder) == 0:
                continue
            folder_text = normalized_text(str(folder))
            if contains_excluded_keyword(folder):
                continue
            if selected:
                if not any(v in folder_text for v in selected):
                    continue
            elif allowed_varieties:
                if not any(v in folder_text for v in allowed_varieties):
                    continue
            elif not matches_target_fruit_or_variety(folder):
                continue

            official_split = "unknown"
            if is_training_path(folder):
                official_split = "training"
            elif is_validation_path(folder):
                official_split = "validation"
            result.append((folder, source_label, official_split))

    return result


def object_id_from_name(image_path: Path) -> str:
    match = OBJECT_ID_PATTERN.search(image_path.name)
    if match:
        return match.group(1)
    return image_path.stem


def split_images_by_object(images: list[Path], ratios: tuple[float, float, float], rng: random.Random) -> dict[str, list[Path]]:
    groups: dict[str, list[Path]] = {}
    for image in images:
        groups.setdefault(object_id_from_name(image), []).append(image)

    group_ids = list(groups.keys())
    rng.shuffle(group_ids)
    n_total = len(group_ids)
    n_train = int(n_total * ratios[0])
    n_valid = int(n_total * ratios[1])
    if n_total >= 3:
        n_valid = max(1, n_valid)
        n_train = min(n_train, n_total - n_valid - 1)

    split_group_ids = {
        "train": group_ids[:n_train],
        "valid": group_ids[n_train : n_train + n_valid],
        "test": group_ids[n_train + n_valid :],
    }
    return {split: [image for group_id in ids for image in groups[group_id]] for split, ids in split_group_ids.items()}


def split_train_to_train_valid(images: list[Path], rng: random.Random) -> dict[str, list[Path]]:
    valid_ratio = TRAIN_TO_VALID_RATIO
    train_ratio = max(0.0, 1.0 - valid_ratio)
    split = split_images_by_object(images, (train_ratio, valid_ratio, 0.0), rng)
    return {"train": split["train"] + split["test"], "valid": split["valid"]}


def count_assignment_images(assignments: dict[str, dict[str, list[Path]]]) -> dict[str, int]:
    return {split: sum(len(images) for images in label_map.values()) for split, label_map in assignments.items()}


def prepare_assignments_from_grade_dirs(grade_dirs: list[tuple[Path, str, str]]) -> dict[str, dict[str, list[Path]]]:
    rng = random.Random(RANDOM_SEED)
    assignments = {split: {label: [] for label in LABELS} for split in ["train", "valid", "test"]}

    for grade_dir, source_label, official_split in grade_dirs:
        target_label = QC_LABEL_MAP[source_label]
        images = sorted(p for p in grade_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        if not images:
            continue

        if USE_OFFICIAL_VALIDATION_AS_TEST and official_split == "training":
            split = split_train_to_train_valid(images, rng)
            assignments["train"][target_label].extend(split["train"])
            assignments["valid"][target_label].extend(split["valid"])
        elif USE_OFFICIAL_VALIDATION_AS_TEST and official_split == "validation":
            assignments["test"][target_label].extend(images)
        else:
            split = split_images_by_object(images, (TRAIN_RATIO, VALID_RATIO, TEST_RATIO), rng)
            for split_name in ["train", "valid", "test"]:
                assignments[split_name][target_label].extend(split[split_name])

    print("Prepared direct-source dataset from original files")
    print("assigned:", count_assignment_images(assignments))
    return assignments


search_roots = []
if IS_KAGGLE:
    if KAGGLE_DATA_ROOT.exists():
        search_roots.append(KAGGLE_DATA_ROOT)
    search_roots.append(KAGGLE_INPUT_ROOT)
search_roots.extend([PROJECT_ROOT / "Data", PROJECT_ROOT / "Data" / "Sample", PROJECT_ROOT, PREPARED_DATA_ROOT])

DATA_ASSIGNMENTS = None
DATA_MODE = "imagefolder"
prepared_root = find_prepared_data_root(search_roots)
if prepared_root is not None:
    DATA_ROOT = prepared_root
    print("Using prepared ImageFolder data:", DATA_ROOT)
else:
    grade_dirs = find_grade_dirs(search_roots)
    if not grade_dirs:
        extracted_roots = extract_relevant_zips(search_roots, EXTRACT_ROOT)
        grade_dirs = find_grade_dirs(search_roots + extracted_roots)

    print("grade dirs found:", len(grade_dirs))
    for folder, source_label, official_split in grade_dirs[:12]:
        print(" ", official_split, source_label, folder, count_images(folder))
    if grade_dirs:
        DATA_ASSIGNMENTS = prepare_assignments_from_grade_dirs(grade_dirs)
        DATA_ROOT = PREPARED_DATA_ROOT
        DATA_MODE = "direct-source"
    else:
        DATA_ROOT = PREPARED_DATA_ROOT
        print("No data was found.")
        print("In Kaggle Notebook > Add data, attach an extracted Training/Validation image dataset.")

print("DATA_MODE:", DATA_MODE)
print("DATA_ROOT:", DATA_ROOT)


## 3. 라벨 기준과 데이터 개수 확인

모델이 예측하는 클래스는 `A/B/C`입니다. 원천 QC 데이터의 `L/M/S`는 아래 기준으로 변환합니다.

| 모델 라벨 | QC 원천 라벨 | 의미 |
|---|---|---|
| A | L | 특 |
| B | M | 상 |
| C | S | 보통 |


In [ ]:
# Step 4: Check image counts by split and label.
def count_split_label_images(split: str, label: str) -> int:
    if DATA_ASSIGNMENTS is not None:
        return len(DATA_ASSIGNMENTS.get(split, {}).get(label, []))
    return count_images(DATA_ROOT / split / TARGET_FRUIT / label)


for split in ["train", "valid", "test"]:
    print(f"[{split}]")
    for label in LABELS:
        print(f"  {label}: {count_split_label_images(split, label)} images")


## 4. 보조 이미지 특징 추출

CNN 등급 예측만 사용하지 않고, 색 선명도, 둥근 정도, 멍 의심 확률을 보조 특징으로 계산합니다.

현재 구현은 Kaggle 기본 환경에서 실행하기 쉽도록 OpenCV 없이 PIL/NumPy만 사용합니다.


In [ ]:
# Step 5: Define PIL/NumPy helper feature functions.
def _load_rgb(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> Image.Image:
    return Image.open(image_path).convert("RGB").resize((image_size, image_size))


def calculate_color_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size).convert("HSV")
    hsv = np.asarray(image, dtype=np.float32)
    saturation = hsv[..., 1] / 255.0
    value = hsv[..., 2] / 255.0
    score = (saturation.mean() * 0.65 + value.mean() * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_roundness_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size)
    arr = np.asarray(image, dtype=np.float32)
    border = np.concatenate(
        [arr[:8].reshape(-1, 3), arr[-8:].reshape(-1, 3), arr[:, :8].reshape(-1, 3), arr[:, -8:].reshape(-1, 3)],
        axis=0,
    )
    bg = np.median(border, axis=0)
    distance = np.linalg.norm(arr - bg, axis=2)
    mask = distance > max(18.0, float(distance.mean()))
    ys, xs = np.where(mask)
    if len(xs) < 50:
        return 50.0

    width = max(1, xs.max() - xs.min() + 1)
    height = max(1, ys.max() - ys.min() + 1)
    aspect_score = min(width, height) / max(width, height)
    fill_ratio = mask.sum() / float(width * height)
    score = (aspect_score * 0.65 + min(fill_ratio / 0.78, 1.0) * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_bruise_probability(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> float:
    image = _load_rgb(image_path, image_size)
    hsv = np.asarray(image.convert("HSV"), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    dark_ratio = np.logical_and(value < 0.33, saturation > 0.18).mean()
    probability = min(1.0, dark_ratio / 0.18)
    return round(float(np.clip(probability, 0.0, 1.0)), 4)


def extract_features(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE) -> dict[str, float]:
    return {
        "color_score": calculate_color_score(image_path, image_size),
        "roundness_score": calculate_roundness_score(image_path, image_size),
        "bruise_probability": calculate_bruise_probability(image_path, image_size),
    }


if DATA_ASSIGNMENTS is not None:
    sample_images = sorted(
        image
        for label_map in DATA_ASSIGNMENTS.values()
        for images in label_map.values()
        for image in images
    )
else:
    sample_images = sorted((DATA_ROOT / "train" / TARGET_FRUIT).rglob("*"))
sample_images = [p for p in sample_images if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
if sample_images:
    print("sample_image:", sample_images[0])
    print(extract_features(sample_images[0]))
else:
    print("No sample images found, skipping feature preview.")


## 5. 신선도 점수와 출고 보조 판단

CNN 예측 등급과 보조 특징을 결합해 `freshness_score`를 계산합니다.

```text
freshness_score =
  CNN 품질 점수 * 0.60
+ 색 선명도 * 0.20
+ 둥근 정도 * 0.10
+ 멍 없음 점수 * 0.10
```


In [ ]:
# 6단계: 신선도 점수 계산과 출고 보조 판단 함수를 정의합니다.
GRADE_SCORES = {
    "A": 90.0,
    "B": 70.0,
    "C": 45.0,
}


@dataclass(frozen=True)
class FreshnessResult:
    fruit_type: str
    model_grade: str
    freshness_score: float
    color_score: float
    roundness_score: float
    bruise_probability: float
    model_decision: str
    model_confidence: float
    model_version: str


def calculate_freshness_score(
    model_grade: str,
    color_score: float,
    roundness_score: float,
    bruise_probability: float,
) -> float:
    grade_score = GRADE_SCORES.get(model_grade, GRADE_SCORES["C"])
    bruise_free_score = max(0.0, min(100.0, (1.0 - bruise_probability) * 100.0))
    score = (
        grade_score * 0.60
        + color_score * 0.20
        + roundness_score * 0.10
        + bruise_free_score * 0.10
    )
    return round(max(0.0, min(100.0, score)), 2)


def make_model_decision(freshness_score: float, bruise_probability: float) -> str:
    if freshness_score < 60.0 or bruise_probability >= 0.5:
        return "HOLD"
    if freshness_score >= 80.0 and bruise_probability < 0.2:
        return "PASS"
    return "REVIEW"


examples = [
    {"grade": "A", "color": 88.2, "roundness": 94.5, "bruise": 0.07},
    {"grade": "B", "color": 72.0, "roundness": 80.0, "bruise": 0.18},
    {"grade": "C", "color": 55.0, "roundness": 68.0, "bruise": 0.52},
]

for item in examples:
    score = calculate_freshness_score(item["grade"], item["color"], item["roundness"], item["bruise"])
    decision = make_model_decision(score, item["bruise"])
    print(item, "=>", score, decision)


## 6. Dataset/DataLoader

If `DATA_MODE == "imagefolder"`, the notebook uses `torchvision.datasets.ImageFolder`.

If `DATA_MODE == "direct-source"`, it uses `DirectImageDataset`, which keeps `(image_path, label)` samples in memory and reads each original image from `/kaggle/input` at batch time. This avoids duplicating the dataset under `/kaggle/working`.


In [ ]:
# Step 7: Define transform, Dataset, DataLoader and inspect train data.
def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE, train: bool = True):
    if train:
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ]
        )

    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )


class DirectImageDataset(Dataset):
    def __init__(self, assignments: dict[str, dict[str, list[Path]]], split: str, transform=None):
        self.classes = list(LABELS)
        self.class_to_idx = {label: idx for idx, label in enumerate(self.classes)}
        self.transform = transform
        self.samples = []
        for label in self.classes:
            for image_path in assignments.get(split, {}).get(label, []):
                self.samples.append((Path(image_path), self.class_to_idx[label]))
        self.targets = [target for _, target in self.samples]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, target = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, target


def dataset_root(split: str, fruit_type: str = TARGET_FRUIT, data_root: str | Path = DATA_ROOT) -> Path:
    return Path(data_root) / split / fruit_type


def build_dataset(
    split: str,
    fruit_type: str = TARGET_FRUIT,
    data_root: str | Path = DATA_ROOT,
    image_size: int = DEFAULT_IMAGE_SIZE,
):
    transform = build_transforms(image_size, train=(split == "train"))
    if DATA_ASSIGNMENTS is not None and Path(data_root) == Path(DATA_ROOT):
        dataset = DirectImageDataset(DATA_ASSIGNMENTS, split=split, transform=transform)
        if len(dataset) == 0:
            raise FileNotFoundError(f"No direct-source images for split: {split}")
        return dataset

    root = dataset_root(split, fruit_type, data_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset folder does not exist: {root}")
    return datasets.ImageFolder(root=str(root), transform=transform)


def build_loader(
    split: str,
    fruit_type: str = TARGET_FRUIT,
    data_root: str | Path = DATA_ROOT,
    image_size: int = DEFAULT_IMAGE_SIZE,
    batch_size: int = 16,
    num_workers: int = 0,
):
    dataset = build_dataset(split, fruit_type, data_root, image_size)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


has_train_images = any(count_split_label_images("train", label) > 0 for label in LABELS)
if has_train_images:
    train_dataset = build_dataset("train")
    print("classes:", train_dataset.classes)
    print("samples:", len(train_dataset))
else:
    print("No train images found, skipping Dataset preview.")


## 7. ResNet18 모델 구성

MVP 모델은 ResNet18입니다. 마지막 fully-connected layer만 `A/B/C` 3개 클래스에 맞게 교체합니다.

Kaggle Internet이 꺼져 있으면 pretrained weight 다운로드가 실패할 수 있습니다. 그 경우 `USE_PRETRAINED = True`로 바꿔 실행합니다.


In [ ]:
# 8단계: ResNet18 모델 생성과 checkpoint 로드 함수를 정의합니다.
def build_model(
    model_name: str = DEFAULT_MODEL_NAME,
    num_classes: int = len(LABELS),
    pretrained: bool = False,
) -> nn.Module:
    if model_name != "resnet18":
        raise ValueError(f"Unsupported model_name: {model_name}")

    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_checkpoint(checkpoint_path: str | Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    labels = checkpoint.get("labels", list(LABELS))
    model = build_model(
        model_name=checkpoint.get("model_name", DEFAULT_MODEL_NAME),
        num_classes=len(labels),
        pretrained=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint


preview_model = build_model(num_classes=len(LABELS), pretrained=False)
print(type(preview_model).__name__)
print("output layer:", preview_model.fc)


## 8. 학습 설정과 실행

Kaggle GPU에서 학습하려면 Notebook 우측 설정에서 Accelerator를 GPU로 켭니다.

기본값은 바로 학습되도록 `RUN_TRAINING = True`, `USE_PRETRAINED = True`입니다. 최종 모델 성능을 우선해 ImageNet 사전학습 가중치로 시작합니다. Kaggle Internet이 꺼져 있어 가중치 다운로드가 실패하면 `USE_PRETRAINED = False`로 바꿔 재실행합니다.


In [ ]:
# 9단계: 학습 설정값을 정의합니다.
RUN_TRAINING = True
USE_PRETRAINED = True

TRAIN_CONFIG = {
    "model_name": "resnet18",
    "image_size": 224,
    "batch_size": 64,
    "epochs": 15,
    "patience": 5,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2 if IS_KAGGLE else 0,
}

TRAIN_CONFIG


In [ ]:
# 10단계: 학습/검증/테스트 루프를 정의하고 best checkpoint를 저장합니다.
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / max(1, len(train_loader))


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            y_true.extend(targets.detach().cpu().tolist())
            y_pred.extend(predicted.detach().cpu().tolist())

    avg_loss = total_loss / max(1, len(data_loader))
    accuracy = accuracy_score(y_true, y_pred) if y_true else 0.0
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0
    return {"loss": avg_loss, "accuracy": accuracy, "f1": f1}


def train_model(
    data_root: Path = DATA_ROOT,
    model_root: Path = MODEL_ROOT,
    fruit_type: str = TARGET_FRUIT,
    model_name: str = DEFAULT_MODEL_NAME,
    model_version: str = DEFAULT_MODEL_VERSION,
    image_size: int = DEFAULT_IMAGE_SIZE,
    batch_size: int = 16,
    epochs: int = 10,
    patience: int = 3,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    num_workers: int = 0,
    pretrained: bool = False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    train_loader = build_loader("train", fruit_type, data_root, image_size, batch_size, num_workers)
    valid_loader = build_loader("valid", fruit_type, data_root, image_size, batch_size, num_workers)
    test_loader = build_loader("test", fruit_type, data_root, image_size, batch_size, num_workers)

    labels = train_loader.dataset.classes
    model = build_model(model_name, num_classes=len(labels), pretrained=pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    model_root.mkdir(parents=True, exist_ok=True)
    best_f1 = -1.0
    best_model_state = None
    best_path = model_root / f"{fruit_type}_{model_name}_best.pt"
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        valid_metrics = evaluate(model, valid_loader, criterion, device)
        print(
            f"epoch={epoch:03d} "
            f"loss={train_loss:.4f} train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['f1']:.4f} "
            f"valid_loss={valid_metrics['loss']:.4f} valid_acc={valid_metrics['accuracy']:.4f} valid_f1={valid_metrics['f1']:.4f}"
        )

        if valid_metrics["f1"] > best_f1:
            best_f1 = valid_metrics["f1"]
            wait = 0
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_model_state,
                    "labels": labels,
                    "fruit_type": fruit_type,
                    "model_name": model_name,
                    "model_version": model_version,
                    "image_size": image_size,
                    "train_config": TRAIN_CONFIG,
                    "valid_metrics": valid_metrics,
                },
                best_path,
            )
            print(f"saved best checkpoint: {best_path}")
        else:
            wait += 1

        if wait >= patience:
            print("Early stopping")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"test_loss={test_metrics['loss']:.4f} "
        f"test_acc={test_metrics['accuracy']:.4f} test_f1={test_metrics['f1']:.4f}"
    )

    checkpoint = torch.load(best_path, map_location="cpu") if best_path.exists() else {}
    checkpoint["test_metrics"] = test_metrics
    if best_path.exists():
        torch.save(checkpoint, best_path)
    return best_path, test_metrics


BEST_CHECKPOINT_PATH = None
TEST_METRICS = None

if RUN_TRAINING and has_train_images:
    try:
        BEST_CHECKPOINT_PATH, TEST_METRICS = train_model(
            data_root=DATA_ROOT,
            model_root=MODEL_ROOT,
            image_size=TRAIN_CONFIG["image_size"],
            batch_size=TRAIN_CONFIG["batch_size"],
            epochs=TRAIN_CONFIG["epochs"],
            patience=TRAIN_CONFIG["patience"],
            lr=TRAIN_CONFIG["learning_rate"],
            weight_decay=TRAIN_CONFIG["weight_decay"],
            num_workers=TRAIN_CONFIG["num_workers"],
            pretrained=USE_PRETRAINED,
        )
    except Exception as exc:
        if USE_PRETRAINED:
            print("pretrained weight 사용 중 오류가 발생했습니다. pretrained=False로 자동 재시도합니다.")
            BEST_CHECKPOINT_PATH, TEST_METRICS = train_model(
                data_root=DATA_ROOT,
                model_root=MODEL_ROOT,
                image_size=TRAIN_CONFIG["image_size"],
                batch_size=TRAIN_CONFIG["batch_size"],
                epochs=TRAIN_CONFIG["epochs"],
                patience=TRAIN_CONFIG["patience"],
                lr=TRAIN_CONFIG["learning_rate"],
                weight_decay=TRAIN_CONFIG["weight_decay"],
                num_workers=TRAIN_CONFIG["num_workers"],
                pretrained=False,
            )
        else:
            raise exc
else:
    print("RUN_TRAINING이 False이거나 train 이미지가 없어 학습을 건너뜁니다.")


## 9. 최종 모델 선정 기준

학습 중에는 validation macro F1이 가장 높아지는 시점의 checkpoint를 best model로 저장합니다.

과일마다 Kaggle Run을 따로 수행하므로, 각 실행 결과에서 아래 기준을 만족하는 checkpoint를 최종 모델로 보관합니다.

1. validation macro F1이 가장 높은 모델
2. validation F1이 비슷하면 test macro F1과 test accuracy가 안정적인 모델
3. train 성능만 높고 valid/test 성능이 낮으면 과적합으로 판단
4. checkpoint에 `labels`, `fruit_type`, `model_name`, `image_size`, `model_version`이 포함되어 FastAPI 추론에 바로 쓸 수 있어야 함

최종 산출물 이름은 `{TARGET_FRUIT}_resnet18_best.pt` 형태입니다.


In [ ]:
# 11단계: 저장된 best checkpoint의 메타데이터를 확인합니다.
candidate_checkpoints = [
    BEST_CHECKPOINT_PATH,
    MODEL_ROOT / f"{TARGET_FRUIT}_resnet18_best.pt",
    WORKING_ROOT / "models" / f"{TARGET_FRUIT}_resnet18_best.pt",
]
candidate_checkpoints = [Path(p) for p in candidate_checkpoints if p is not None]
checkpoint_path = next((p for p in candidate_checkpoints if p.exists()), None)

if checkpoint_path is None:
    print("아직 확인 가능한 checkpoint가 없습니다. 학습 완료 후 다시 실행하세요.")
else:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    print("selected_checkpoint:", checkpoint_path)
    print("model_name:", checkpoint.get("model_name"))
    print("model_version:", checkpoint.get("model_version"))
    print("labels:", checkpoint.get("labels"))
    print("image_size:", checkpoint.get("image_size"))
    print("valid_metrics:", checkpoint.get("valid_metrics"))
    print("test_metrics:", checkpoint.get("test_metrics"))


## 10. 단일 이미지 추론

최종 checkpoint와 이미지 1장을 입력하면 CNN 예측 등급, confidence, 보조 특징, 신선도 점수, 출고 보조 판단을 계산합니다.


In [ ]:
# 12단계: 단일 이미지 추론 함수를 정의하고, 샘플 이미지가 있으면 바로 실행합니다.
def predict_image(
    image_path: str | Path,
    checkpoint_path: str | Path,
    fruit_type: str = TARGET_FRUIT,
    device_name: str | None = None,
) -> FreshnessResult:
    device = torch.device(device_name or ("cuda" if torch.cuda.is_available() else "cpu"))
    model, checkpoint = load_checkpoint(checkpoint_path, device)
    labels = checkpoint["labels"]
    image_size = int(checkpoint.get("image_size", DEFAULT_IMAGE_SIZE))
    model_version = checkpoint.get("model_version", DEFAULT_MODEL_VERSION)

    transform = build_transforms(image_size=image_size, train=False)
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1)[0]

    confidence, index = torch.max(probabilities, dim=0)
    model_grade = labels[int(index.item())]
    model_confidence = round(float(confidence.item()), 4)

    features = extract_features(image_path, image_size)
    freshness_score = calculate_freshness_score(
        model_grade=model_grade,
        color_score=features["color_score"],
        roundness_score=features["roundness_score"],
        bruise_probability=features["bruise_probability"],
    )
    model_decision = make_model_decision(freshness_score, features["bruise_probability"])

    return FreshnessResult(
        fruit_type=fruit_type,
        model_grade=model_grade,
        freshness_score=freshness_score,
        color_score=features["color_score"],
        roundness_score=features["roundness_score"],
        bruise_probability=features["bruise_probability"],
        model_decision=model_decision,
        model_confidence=model_confidence,
        model_version=model_version,
    )


if checkpoint_path is not None and sample_images:
    result = predict_image(sample_images[0], checkpoint_path)
    print(asdict(result))
else:
    print("checkpoint 또는 샘플 이미지가 없어 추론 예시는 건너뜁니다.")


## 11. FastAPI 응답 스펙과 연동 준비

FastAPI에서는 `predict_image()` 결과를 Docs의 API 설계서 형식에 맞춰 아래 JSON 형태로 변환하면 됩니다.

문서 기준 공통 응답 구조는 `{ "data": ..., "message": "success", "error": null }`입니다. 품질 검사 저장 컬럼과 맞추기 위해 모델 예측 등급은 `model_grade`, 모델 판정은 `model_decision` 필드명을 사용합니다.


In [ ]:
# 13단계: FastAPI 응답 변환 함수를 정의합니다.
# Docs/09_API_설계서의 quality_inspections 응답 필드명에 맞춥니다.
def to_api_response(result: FreshnessResult, quality_inspection_id: int | None = None) -> dict[str, Any]:
    return {
        "data": {
            "quality_inspection_id": quality_inspection_id,
            "model_grade": result.model_grade,
            "freshness_score": result.freshness_score,
            "color_score": result.color_score,
            "roundness_score": result.roundness_score,
            "bruise_probability": result.bruise_probability,
            "model_decision": result.model_decision,
            # 아래 값들은 문서 필수 필드는 아니지만 서버 로그/디버깅/모델 관리에 유용합니다.
            "fruit_type": result.fruit_type,
            "model_confidence": result.model_confidence,
            "model_version": result.model_version,
        },
        "message": "success",
        "error": None,
    }


mock_result = FreshnessResult(
    fruit_type=TARGET_FRUIT,
    model_grade="A",
    freshness_score=91.3,
    color_score=88.2,
    roundness_score=94.5,
    bruise_probability=0.07,
    model_decision="PASS",
    model_confidence=0.86,
    model_version=DEFAULT_MODEL_VERSION,
)

to_api_response(mock_result, quality_inspection_id=31)


## 최종 정리

이 노트북은 Kaggle Notebook에 **노트북 파일만 import하고, 데이터셋만 Add data로 연결하면** 실행 가능한 구조입니다.

과일마다 한 번씩 별도로 실행합니다. 실행할 때 상단 설정 블록에서 아래 값만 현재 데이터셋에 맞게 바꾸면 됩니다.

```python
TARGET_FRUIT = "apple"        # apple / pear / mandarine / persimmon
SELECTED_VARIETIES = []        # 비우면 참고1 PDF의 해당 과일 대상 품종 전체 사용
OBJECT_ID_REGEX = r"..."
```

Kaggle에서 확인할 것은 두 가지입니다.

1. Notebook Accelerator를 GPU로 설정
2. Add data로 현재 과일의 원천 폴더 또는 `train/valid/test/{TARGET_FRUIT}/A-B-C` 구조의 데이터셋 연결

최종적으로 과일별 실행 결과에서 checkpoint를 하나씩 확보합니다.

```text
apple_resnet18_best.pt
pear_resnet18_best.pt
mandarine_resnet18_best.pt
persimmon_resnet18_best.pt
```

Docs의 `12_DL_신선도판별_기획서.md`에서는 산출물을 `freshness_model.pt`로 표현하지만, 현재 구현은 과일별 성능과 특징 차이를 고려해 같은 구조의 checkpoint를 과일별로 저장합니다. FastAPI에서는 `fruit_type`에 따라 알맞은 checkpoint를 선택해 로드합니다.
